In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dateutil import parser
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import initializers
from tensorflow.keras import regularizers
from keras.layers import Dense, SimpleRNN, LSTM ,GRU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from sklearn.metrics import mean_squared_error,mean_absolute_percentage_error,mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

spread = pd.read_csv("../data/yield/spread.csv")
spread = spread[["Date", "US_KR_3Y", "US_KR_10Y"]]
spread = spread=spread.sort_values("Date").ffill()


In [ ]:
def form_arrays(x, lookback=3, delay=1, step=1, feature_columns=[0], target_columns=[0], 
                unique=False, verbose=False, predict="scalar"):
    """
    Professor's function to form time-series mini-batches
    """
    i_start = 0
    count = 0
    
    x_out = []
    y_out = []
    
    while i_start + lookback + delay < x.shape[0]:
        i_stop = i_start + lookback
        i_pred = i_stop + delay
        
        if verbose and count < 2:
            print("indice range:", i_start, i_stop, "-->", i_pred)
        
        indices_to_keep = []
        j = i_stop
        while j >= i_start:
            indices_to_keep.append(j)
            j = j - step
        
        xtmp = x[indices_to_keep, :]
        xtmp = xtmp[:, feature_columns]
        
        if predict == "scalar":
            ytmp = x[i_pred, target_columns]
        if predict == "vector":
            ytmp = x[i_stop + 1:i_pred, target_columns]
        
        x_out.append(xtmp)
        y_out.append(ytmp)
        
        if verbose and count < 2:
            print("X:\n", xtmp, "\nY:\n", ytmp)
            print("shape:", xtmp.shape, "-->", ytmp.shape)
        
        if verbose and count < 2:
            fig, ax = plt.subplots()
            ax.plot(x, 'b-')
            ax.plot(x, 'bx')
            ax.plot(indices_to_keep, xtmp, 'go')
            if predict == "scalar":
                ax.plot(i_pred * np.ones(len(target_columns)), ytmp, 'ro')
            elif predict == "vector":
                ax.plot(i_stop + 1 + np.tile(np.arange(ytmp.shape[0]), (ytmp.shape[1], 1)), 
                       ytmp.T, 'ro')
            plt.show()
        
        if unique:
            i_start += lookback
        i_start += 1
        count += 1
    
    return np.array(x_out), np.array(y_out)


In [ ]:
def regression_report(yt, ytp, yv, yvp):
    """
    Professor's regression reporting function
    """
    print("---------- Regression report ----------")
    print("TRAINING:")
    print(" MSE:", mean_squared_error(yt, ytp))
    print(" MAE:", mean_absolute_error(yt, ytp))
    print(" RMSE:", np.sqrt(mean_squared_error(yt, ytp)))
    
    # PARITY PLOT
    fig, ax = plt.subplots()
    ax.plot(yt, ytp, 'ro')
    ax.plot(yt, yt, 'b-')
    ax.set(xlabel='y_data', ylabel='y_predicted',
           title='Training data parity plot (line y=x represents a perfect fit)')
    plt.show()
    
    # PLOT PART OF THE PREDICTED TIME-SERIES
    frac_plot = 1.0
    upper = int(frac_plot * yt.shape[0])
    fig, ax = plt.subplots()
    ax.plot(yt[0:upper], 'b-')
    ax.plot(ytp[0:upper], 'r-', alpha=0.5)
    ax.plot(ytp[0:upper], 'ro', alpha=0.25)
    ax.set(xlabel='index', ylabel='y(t) (blue=actual & red=prediction)', 
           title='Training: Time-series prediction')
    plt.show()
    
    print("VALIDATION:")
    print(" MSE:", mean_squared_error(yv, yvp))
    print(" MAE:", mean_absolute_error(yv, yvp))
    print(" RMSE:", np.sqrt(mean_squared_error(yv, yvp)))
    
    # PARITY PLOT
    fig, ax = plt.subplots()
    ax.plot(yv, yvp, 'ro')
    ax.plot(yv, yv, 'b-')
    ax.set(xlabel='y_data', ylabel='y_predicted',
           title='Validation data parity plot (line y=x represents a perfect fit)')
    plt.show()
    
    # PLOT PART OF THE PREDICTED TIME-SERIES
    upper = int(frac_plot * yv.shape[0])
    fig, ax = plt.subplots()
    ax.plot(yv[0:upper], 'b-')
    ax.plot(yvp[0:upper], 'r-', alpha=0.5)
    ax.plot(yvp[0:upper], 'ro', alpha=0.25)
    ax.set(xlabel='index', ylabel='y(t) (blue=actual & red=prediction)', 
           title='Validation: Time-series prediction')
    plt.show()


def history_plot(history):
    """
    Professor's history plotting function
    """
    FS = 16
    history_dict = history.history
    loss_values = history_dict["loss"]
    val_loss_values = history_dict["val_loss"]
    epochs = range(1, len(loss_values) + 1)
    plt.plot(epochs, loss_values, "bo", label="Training loss")
    plt.plot(epochs, val_loss_values, "b", label="Validation loss")
    plt.title("Training and validation loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()


In [ ]:
def forecast(model, last_input, n_steps):
    """
    Professor's forecasting function
    """
    forecast_vals = []
    input_seq = last_input.copy().reshape(1, *last_input.shape)
    for _ in range(n_steps):
        pred = model.predict(input_seq, verbose=0)[0, 0]
        forecast_vals.append(pred)
        next_input = np.append(input_seq[0, 1:], [[pred]], axis=0)
        input_seq = next_input.reshape(1, *next_input.shape)
    return forecast_vals

In [ ]:
def prepare_data(series, lookback=30, train_split=0.7, val_split=0.2):
    """
    Prepare data using professor's form_arrays function
    """
    # Convert to numpy array
    data = series.values.reshape(-1, 1)
    
    # Normalize
    scaler = MinMaxScaler(feature_range=(0, 1))
    data_scaled = scaler.fit_transform(data)
    
    # Form arrays using professor's function
    X, y = form_arrays(
        data_scaled,
        lookback=lookback,
        delay=1,
        step=1,
        feature_columns=[0],
        target_columns=[0],
        unique=False,
        verbose=False,
        predict="scalar"
    )
    
    print(f"Total samples: {len(X)}")
    print(f"X shape: {X.shape}, y shape: {y.shape}")
    